# App

> The main fasthtml app of solveblog

In [ ]:
#|default_exp app

In [ ]:
import tempfile, sys
from solveblog.core import solveblog_new

In [ ]:
tmp = tempfile.mkdtemp(prefix='solveblog_dev_')

In [ ]:
%cd {tmp}
solveblog_new(name='Test User', 
              port='8000', 
              email='test@test.com',
              description='A test blog', 
              github='gh-user',
              twitter='x-user', 
              linkedin='li-user')


/tmp/solveblog_dev_zp56_1yz
✓ solveblog project created! Next step run: `!solveblog` to start your blog.


In [ ]:
if '' not in sys.path: sys.path.insert(0, '')

## Setup

In [ ]:
#| export
from fastcore.nbio import *
from fastcore.utils import *
import json, os, re, mistletoe as mst
from functools import lru_cache,cache
from collections import Counter
from dateutil.parser import parse as parse_date
from fastcore.ansi import ansi2html
from html import escape
from datetime import datetime, date
from fasthtml.common import *
from fastlite import *
from monsterui.all import *
from urllib.parse import quote, unquote
from solveblog import load_cfg

In [ ]:
from fastcore.test import *

In [ ]:
#| export
cfg = load_cfg()
cfg.dom = 'https://'+json.loads(os.environ['PUBLIC_DOMAINS'])[str(cfg.port)]

## Database

In [ ]:
#| export
db = database("subscribers.db")
class Subscriber: id:int; email:str; created_at:str
subscribers = db.create(Subscriber, transform=True)

## Notebook Rendering

In [ ]:
#| export
def nb2md(path):
    "Read notebook, return frontmatter Post object (like frontmatter.load for .md files)"
    cells = json.loads(Path(path).read_text()).get('cells', [])
    fm_src, md_cells = '', cells
    if cells and cells[0]['cell_type'] == 'raw':
        fm_src = ''.join(cells[0]['source'])
        md_cells = cells[1:]
    def cell_md(c):
        src = ''.join(c['source'])
        if c.get('metadata', {}).get('pinned'): return None
        if c['cell_type'] == 'code':
            code = f'<pre><code class="language-python">{escape(src)}</code></pre>'
            if outs := render_outputs(c.get('outputs', [])): return f'<div class="cell">{code}<div class="cell-output">{outs}</div></div>'
            return f'<div class="cell">{code}</div>'
        if c['cell_type'] == 'raw': return f'```\n{src}\n```'
        if c['cell_type'] == 'markdown':
            if atts := c.get('attachments', {}):
                for fname, mimes in atts.items():
                    mime, data = next(iter(mimes.items()))
                    src = src.replace(f'attachment:{fname}', f'data:{mime};base64,{data}')
            if c.get('metadata', {}).get('solveit_ai'):
                sep = re.split(r'##### 🤖Reply🤖<!--.*?-->\n*', src, maxsplit=1)
                prompt = sep[0].strip()
                if len(sep) > 1 and sep[1].strip():
                    response = sep[1].strip()
                    return f'<div class="ai-cell"><div class="ai-prompt"><span class="ai-label">PROMPT</span>\n\n{prompt}\n\n</div><div class="ai-response"><span class="ai-label">AI</span>\n\n{response}\n\n</div></div>'
                return f'<div class="ai-cell"><div class="ai-prompt"><span class="ai-label">PROMPT</span>\n\n{prompt}\n\n</div></div>'
            return src
    rendered = [r for r in (cell_md(c) for c in md_cells) if r is not None]
    md = (rendered[0] + '\n\n<!--INTRO-->\n\n' + '\n\n'.join(rendered[1:])) if len(rendered) > 1 else '\n\n'.join(rendered)
    return frontmatter(f"{fm_src}\n\n{md}") if fm_src else ({}, md)

In [ ]:
fm,md = nb2md('public/posts/hello-world.ipynb')

In [ ]:
test_eq(fm['title'],'Test Dialog Post')
test_eq(fm['date'],date(2026,4,2))
test_eq(fm['tags'],['test'])

In [ ]:
assert '<!--INTRO-->' in md

## Markdown Rendering

In [ ]:
#| export
def span_token(name, pat, attr, prec=5):
    class T(mst.span_token.SpanToken):
        precedence,parse_inner,parse_group,pattern = prec,False,1,re.compile(pat)
        def __init__(self, match): setattr(self, attr, match.group(1))
    T.__name__ = name
    return T

In [ ]:
#| export
FootnoteRef = span_token('FootnoteRef', r'\[\^([^\]]+)\](?!:)', 'target')
YoutubeEmbed = span_token('YoutubeEmbed', r'\[yt:([a-zA-Z0-9_-]+)\]', 'video_id', 6)

In [ ]:
#| export
def extract_footnotes(content):
    pat = re.compile(r'^\[\^([^\]]+)\]:\s*(.+?)(?=(?:^|\n)\[\^|\n\n|\Z)', re.MULTILINE | re.DOTALL)
    defs = {m.group(1): m.group(2).strip() for m in pat.finditer(content)}
    for m in pat.finditer(content): content = content.replace(m.group(0), '', 1)
    return content.strip(), defs

In [ ]:
#| export
class ContentRenderer(FrankenRenderer):
    def __init__(self, *extras, img_dir=None, footnotes=None, **kwargs):
        super().__init__(*extras, img_dir=img_dir, **kwargs)
        self.footnotes,self.fn_counter = footnotes or {},0
    
    def render_footnote_ref(self, token):
        self.fn_counter += 1
        n,target = self.fn_counter,token.target
        content = self.footnotes.get(target, f"[Missing footnote: {target}]")
        rendered = mst.markdown(content, partial(ContentRenderer, img_dir=self.img_dir)).strip()
        if rendered.startswith('<p>') and rendered.endswith('</p>'): rendered = rendered[3:-4]
        style = "text-sm leading-relaxed border-l-2 border-amber-400 dark:border-blue-400 pl-3 text-neutral-500 dark:text-neutral-400 transition-all duration-500 w-full my-2 xl:my-0"
        toggle_hs = f"on click if window.innerWidth >= 1280 then add .hl to #sn-{n} then wait 1s then remove .hl from #sn-{n} else toggle .open on me then toggle .show on #sn-{n}"
        ref = Span(id=f"snref-{n}", role="doc-noteref", aria_label=f"Sidenote {n}", cls="sidenote-ref cursor-pointer", _=toggle_hs)
        note = Span(NotStr(rendered), id=f"sn-{n}", role="doc-footnote", aria_labelledby=f"snref-{n}", cls=f"sidenote {style}")
        hide = lambda c: to_xml(Span(c, cls="hidden", aria_hidden="true"))
        return hide(" (") + to_xml(ref) + to_xml(note) + hide(")")
    
    def render_youtube_embed(self, token):
        iframe = Iframe(src=f"https://www.youtube.com/embed/{token.video_id}", title="YouTube video player", frameborder="0",
                        allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share",
                        referrerpolicy="strict-origin-when-cross-origin", allowfullscreen=True, cls="w-full aspect-video rounded-lg my-6")
        return to_xml(iframe)
    
    def render_link(self, token):
        href,inner,title = token.target,self.render_inner(token),f' title="{token.title}"' if token.title else ''
        is_internal = href.startswith('/') and not href.startswith('//') and '.' not in href.split('/')[-1]
        hx = f' hx-get="{href}" hx-target="#main-content" hx-push-url="true" hx-swap="innerHTML show:window:top"' if is_internal else ''
        ext = '' if is_internal else ' target="_blank" rel="noopener noreferrer"'
        return f'<a href="{href}"{hx}{ext} class="text-primary underline"{title}>{inner}</a>'

In [ ]:
#| export
def from_md(content, img_dir='/public/posts'):
    content, footnotes = extract_footnotes(content)
    mods = {'pre': 'border border-gray-300 rounded-md my-4', 'p': 'text-base leading-relaxed mb-6', 'li': 'text-base leading-relaxed',
            'ul': 'uk-list uk-list-bullet space-y-2 mb-6 ml-6 text-base', 'ol': 'uk-list uk-list-decimal space-y-2 mb-6 ml-6 text-base', 'hr': 'border-t border-border my-8'}
    rendered = render_md(content, class_map_mods=mods, img_dir=img_dir, renderer=partial(ContentRenderer, FootnoteRef, YoutubeEmbed, footnotes=footnotes))
    return Div(rendered, cls="w-full")

## App init

In [ ]:
#| export
sidenote_css = Style("""
.sidenote-ref { 
    color: rgb(221, 166, 55); background-color: rgba(221, 166, 55, 0.15);
    display: inline-block; transition: transform 0.2s;
    font-size: 0.6rem; vertical-align: top; line-height: 1;
    padding: 0 0.15rem; border-radius: 0.2rem; margin-left: 0.1rem;
}
.sidenote-ref:after { content: "›"; }
.dark .sidenote-ref { color: rgb(96, 165, 250); background-color: rgba(96, 165, 250, 0.15); }
.sidenote { display: none; }
@media (max-width: 1279px) {
    .sidenote-ref:after { content: "⌃"; }
    .sidenote-ref { transform: rotate(180deg); }
    .sidenote-ref.open { transform: rotate(0deg); }
    .sidenote.show { display: block; float: left; clear: both; width: 95%; margin: 0.75rem 2.5%; position: relative; }
}
@media (min-width: 1280px) {
    .sidenote { display: block; float: right; clear: right; width: 14rem; margin-right: -16rem; margin-top: 0.25rem; margin-bottom: 0.75rem; }
    .sidenote.hl { background-color: rgba(221, 166, 55, 0.1); }
    .dark .sidenote.hl { background-color: rgba(96, 165, 250, 0.1); }
}
""")

In [ ]:
#| export
cell_css = Style("""
.cell pre { margin-bottom: 0; border-bottom-left-radius: 0; border-bottom-right-radius: 0; }
.cell-output pre, .cell-output pre code { background: #f0f0f0; }
.dark .cell-output pre, .dark .cell-output pre code { background: #1e2128; }
.cell-output { border: 1px solid rgb(209 213 219); border-top: 0; border-bottom-left-radius: 0.375rem; border-bottom-right-radius: 0.375rem; overflow: hidden; }
.ansi-red-fg { color: #e75c58; } .ansi-green-fg { color: #00a250; } .ansi-yellow-fg { color: #ddb62b; }
.ansi-blue-fg { color: #208ffb; } .ansi-magenta-fg { color: #d160c4; } .ansi-cyan-fg { color: #60c6c8; }
.ansi-white-fg { color: #c5c1b4; } .ansi-bold { font-weight: bold; }
.ai-cell { border: 1px solid rgb(209 213 219); border-radius: 0.375rem; overflow: hidden; margin: 1.5rem 0; }
.ai-prompt { padding: 0.75rem 1rem; }
.ai-prompt > p:last-child, .ai-response > p:last-child { margin-bottom: 0; }
.ai-response { padding: 0.75rem 1rem; background: #f0f0f0; border-top: 1px solid rgb(209 213 219); }
.dark .ai-response { background: #1e2128; }
.ai-prompt .ai-label, .ai-response .ai-label { float: right; font-size: 0.6rem; font-weight: 600; color: rgb(156 163 175); letter-spacing: 0.05em; margin-left: 0.5rem; }
""")

In [ ]:
#| export
_custom_css = Style(Path('custom.css').read_text()) if Path('custom.css').exists() else ''

hdrs = (*Theme.slate.headers(highlightjs=True),
        Link(rel="icon", href="/public/favicon.ico"),
        Script(src="https://unpkg.com/hyperscript.org@0.9.12"),
        Link(rel="preconnect", href="https://fonts.googleapis.com"), Link(rel="preconnect", href="https://fonts.gstatic.com", crossorigin=""),
        Link(rel="stylesheet", href="https://fonts.googleapis.com/css2?family=IBM+Plex+Sans:wght@400;500;600;700&family=IBM+Plex+Mono&display=swap"),
        Style("body { font-family: 'IBM Plex Sans', sans-serif; } code, pre { font-family: 'IBM Plex Mono', monospace; }"),
        sidenote_css, cell_css, _custom_css)

app = FastHTML(hdrs=hdrs)
app.mount("/public", StaticFiles(directory="public"), name="public")
rt = app.route


In [ ]:
#| export
def theme_toggle():
    return Button(UkIcon("moon", cls="dark:hidden"), UkIcon("sun", cls="hidden dark:block"),
        _="""on click
                toggle .dark on <html/>
                set franken to (localStorage's __FRANKEN__ or '{}') as Object
                if the first <html/> matches .dark set franken's mode to 'dark' else set franken's mode to 'light' end
                set localStorage's __FRANKEN__ to franken as JSON""",
        cls="p-1 hover:scale-110 shadow-none", type="button")

In [ ]:
#| export
def x_icon(): return Svg(ft_hx("path", d="M12.6.75h2.454l-5.36 6.142L16 15.25h-4.937l-3.867-5.07-4.425 5.07H.316l5.733-6.57L0 .75h5.063l3.495 4.633L12.601.75Zm-.86 13.028h1.36L4.323 2.145H2.865z"), width=20, height=20, fill="currentColor", viewBox="0 0 16 16", aria_hidden="true")

In [ ]:
#| export
def social_link(k, v):
    ext = dict(rel="nofollow noindex") if k == "mail" else {} if k == "rss" else dict(target="_blank", rel="noopener noreferrer")
    return A(x_icon() if k == "twitter" else UkIcon(k, width=20, height=20), href=v, aria_label=k.title(), cls="hover:text-primary transition-colors", **ext)

In [ ]:
#| export
_social_urls = dict(github='https://github.com/{}', twitter='https://x.com/{}', linkedin='https://www.linkedin.com/in/{}/')
socials = {k: _social_urls[k].format(cfg[k]) for k in _social_urls if cfg.get(k)}
socials['rss'] = '/rss.xml'
if cfg.get('email'): socials['mail'] = '/contact'
ftr_content = Div(*[social_link(k,v) for k,v in socials.items()], cls="flex justify-center gap-6 text-muted-foreground")

In [ ]:
#| export
def read(p): return nb2md(p) if p.suffix=='.ipynb' else frontmatter(p.read_text())

In [ ]:
#| export
pages = {p.stem.split('_',1)[1]: read(p) for p in Path('public/pages').ls(file_exts=['.md','.ipynb']).sorted()}

In [ ]:
#| export
def hx_attrs(url, target="#main-content"): return dict(hx_get=url, hx_target=target, hx_push_url="true", hx_swap="innerHTML show:window:top")
def hx_link(txt, href, cls="text-primary underline", target="#main-content", **kw): return A(txt, cls=cls, **hx_attrs(href, target), **kw)

In [ ]:
#| export
def navbar():
    menu_id,btn_id = f"menu-{unqid()}",f"btn-{unqid()}"
    brand = A(Img(src="/public/nav.png", alt=cfg.name, cls="w-6 h-6 rounded-full"), Span(cfg.name), href="/", cls="flex items-center gap-2 text-lg font-bold", **hx_attrs("/"))
    def navlinks(_=None): return [hx_link("Blog", "/blog", cls="hover:scale-110", _=_), *[hx_link(s.title(), f"/{s}", cls="hover:scale-110", _=_) for s in pages]]
    hamburger = Button(UkIcon("menu", width=30, height=30), cls="p-0 border-0 shadow-none", _=f"on click toggle .hidden on #{menu_id}", type="button", id=btn_id)
    return Nav(cls="border rounded-lg shadow backdrop-blur-md bg-background/98")(
            Div(brand, Div(*navlinks(), theme_toggle(), cls="hidden md:flex items-center space-x-4 ml-auto"),
                Div(theme_toggle(), hamburger, cls="flex md:hidden items-center gap-4 ml-auto"), cls="flex items-center p-4"),
            Div(*navlinks(_=f"on click add .hidden to #{menu_id}"), cls="flex flex-col space-y-4 p-4 text-center hidden", id=menu_id, _=f"on click elsewhere if event.target.closest('#{btn_id}') is null add .hidden to me")
    )

In [ ]:
#| export
def layout(*content, htmx, title=None):
    if htmx and htmx.request: return (Title(title), *content)
    main = Main(*content, cls='w-full max-w-2xl mx-auto px-6 py-8 space-y-8', id="main-content")
    ctr_cls = 'w-full max-w-2xl mx-auto'
    return Title(title), Div(cls="flex flex-col min-h-screen")(
        Div(navbar(), cls=f'{ctr_cls} px-4 sticky top-0 z-50 mt-4'),
        main,
        Footer(Divider(), subscribe_form(), ftr_content, cls=f'{ctr_cls} px-6 mt-auto mb-6 space-y-6')
    )

In [ ]:
#| export
def subscribe_form():
    return Div(
        P("Subscribe via ", A("RSS", href="/rss.xml", cls="text-primary underline"), " or enter your email to get notified of new posts directly in your inbox", cls="text-sm text-muted-foreground mb-2"),
        Form(Input(type="email", name="email", placeholder="your@email.com", required=True, cls="flex-1 rounded-l-md"), Button("Subscribe", cls=(ButtonT.primary, "rounded-l-none rounded-r-md")), cls="flex", hx_post="/subscribe", hx_swap="outerHTML"),
        cls="mt-6")

## Posts & Tags

In [ ]:
#| export
class Post:
    def __init__(self, path):
        self.path,self.slug = (p := Path(path)),p.stem
        self.meta, self.content = read(p)
        self.title,self.excerpt,self.tags = self.meta['title'],self.meta.get('excerpt',''),L(self.meta.get('tags', []))
        self.date = d if isinstance(d:=self.meta['date'], date) else parse_date(str(d)).date()
        self.datestr = self.date.strftime('%d %b %Y')
        self.external_url = self.meta.get('external_url')
        self.unlisted = self.meta.get('unlisted', False)
        self.img_dir = '/'+str(self.path.parent)

In [ ]:
#| export
posts = {p.slug: p for p in L(Path('public/posts').rglob('*')).filter(lambda p: p.suffix in ('.md','.ipynb') and not any(part.startswith('.') for part in p.parts)).map(Post).sorted(key=lambda p: p.date, reverse=True)}

In [ ]:
#| export
_pill_cls = "text-xs px-2 py-1 rounded border transition-all"
def tag_label(tag): return Span(tag, cls=f"{_pill_cls} bg-muted")

def tag_link(tag, target="#main-content"): return A(tag, cls=f"{_pill_cls} bg-muted hover:bg-primary/20", **hx_attrs(f"/blog?tags={quote(tag)}", target))

def tag_toggle(tag, selected, avail=None):
    new = ','.join(quote(t) for t in selected ^ {tag})
    hx = hx_attrs(f"/blog?tags={new}" if new else "/blog", "#posts-list")
    if tag in selected:        return A(tag, cls=f"{_pill_cls} bg-primary cursor-pointer text-primary-foreground", **hx)
    if avail and tag in avail: return A(tag, cls=f"{_pill_cls} cursor-pointer bg-muted hover:bg-primary/20", **hx)
    return Span(tag, cls=f"{_pill_cls} opacity-30 cursor-not-allowed")

In [ ]:
#| export
def tag_filter(selected, all_posts, filtered):
    counts = Counter(all_posts.attrgot('tags').concat())
    if not counts: return Div(id="tag-filter")
    avail = set(filtered.attrgot('tags').concat())
    tags = sorted(counts, key=lambda t: (-counts[t], t))
    btns = [tag_toggle(t, selected, avail) for t in tags]
    if selected: btns.append(ft_hx("button", UkIcon("x", width=12, height=12), "Clear",
                           cls=f"{_pill_cls} flex items-center gap-1 cursor-pointer hover:shadow-sm", **hx_attrs("/blog", "#posts-list")))
    return Div(Span("Filter:", cls="text-sm font-medium mr-1"), *btns, cls="pb-4 flex flex-wrap items-center gap-2", id="tag-filter")

In [ ]:
#| export
def post_intro(content):
    if '<!--INTRO-->' in content: return content.split('<!--INTRO-->', 1)[0].rstrip()
    m = re.search(r'^##\s', content, re.MULTILINE)
    return content[:m.start()].rstrip() if m else content


In [ ]:
#| export
def post_preview(p):
    intro, url = post_intro(p.content), blogpost.to(slug=p.slug)
    return Div(
        Div(Span(cls="flex-1 border-t border-muted"), Span(p.datestr, cls="text-sm text-muted-foreground px-4"), Span(cls="flex-1 border-t border-muted"), cls="flex items-center my-8"),
        H2(hx_link(p.title, url, cls="font-bold")), from_md(intro, img_dir=p.img_dir),
        Div(hx_link("↪ Keep reading", url, cls="text-sm text-primary hover:underline") if intro != p.content else Span(),
            Div(*p.tags.map(partial(tag_link, target="#posts-list")), cls="flex gap-2 flex-wrap"), cls="flex justify-between items-center mt-4"))

In [ ]:
#| export
def post_article(p, slug):
    content = p.content
    tags = Div(*p.tags.map(tag_link), cls="flex gap-2 flex-wrap") if p.tags else None
    return Article(H1(p.title, cls="text-3xl font-bold mb-3"),
                   Div(Span(p.date.strftime("%B %d, %Y"), cls="text-muted-foreground text-sm"), tags, cls="flex justify-between items-center mb-8 flex-wrap gap-4"),
                   from_md(content, img_dir=p.img_dir), cls="mb-8")

## Routes

In [ ]:
#| export
@rt
def index(): return RedirectResponse('/blog', status_code=307)

In [ ]:
#| export
@rt
def blog(htmx, tags:str=None):
    selected = {unquote(t.strip()) for t in (tags or '').split(',') if t.strip()}
    all_posts = L(posts.values()).filter(lambda p: not p.unlisted)
    filtered = all_posts.filter(lambda p: selected <= set(p.tags))
    posts_content = Div(*filtered.map(post_preview)) if filtered else Div(P("No posts found matching those tags.", cls="text-muted-foreground"), cls="py-8 text-center")
    posts_div = Div(posts_content, id="posts-list")
    tag_filt = tag_filter(selected, all_posts, filtered)
    if htmx and htmx.target == "posts-list":
        tag_filt.attrs['hx-swap-oob'] = 'true'
        return posts_div, tag_filt
    return layout(H2("Blog"), tag_filt, posts_div, title=f"{cfg.name} - Blog", htmx=htmx)

In [ ]:
#| export
@rt('/blog/{slug}')
def blogpost(htmx, slug:str):
    p = posts.get(slug)
    if not p: return layout(H1("Post Not Found", cls="text-4xl font-bold mb-4"), P("Sorry, this blog post doesn't exist."), title="Post Not Found", htmx=htmx)
    if p.external_url: return Response(headers={"HX-Redirect": p.external_url})
    return layout(post_article(p, slug), title=f"{cfg.name} - {p.title}", htmx=htmx)


In [ ]:
#| export
@rt
def subscribe(email:str):
    if list(subscribers.rows_where("email = ?", [email], limit=1)): return P("You're already subscribed!", cls="text-sm text-muted-foreground mt-6")
    subscribers.insert(Subscriber(email=email, created_at=datetime.now().isoformat()))
    return P("Thanks for subscribing!", cls="text-sm text-green-600 mt-6")

In [ ]:
#| export
@rt('/rss.xml')
def rss_feed():
    ps = list(posts.values())
    def item(p): return f"<item><title>{p.title}</title><link>{cfg.dom}/blog/{p.slug}</link><guid>{cfg.dom}/blog/{p.slug}</guid><pubDate>{p.date.strftime('%a, %d %b %Y %H:%M:%S +0000')}</pubDate><description><![CDATA[{p.excerpt}]]></description>{''.join(f'<category>{t}</category>' for t in p.tags)}</item>"
    items = ''.join(item(p) for p in ps if not p.external_url)
    rss = f'<?xml version="1.0" encoding="UTF-8"?><rss version="2.0" xmlns:atom="http://www.w3.org/2005/Atom"><channel><title>{cfg.name}</title><link>{cfg.dom}</link><description>{cfg.description}</description><language>en-us</language><lastBuildDate>{ps[0].date.strftime("%a, %d %b %Y %H:%M:%S +0000") if ps else ""}</lastBuildDate><atom:link href="{cfg.dom}/rss.xml" rel="self" type="application/rss+xml"/>{items}</channel></rss>'
    return Response(rss, media_type="application/rss+xml")

In [ ]:
#| export
if cfg.get('email'):
    @rt
    def contact(): return RedirectResponse(f"mailto:{cfg.email}", status_code=302)


In [ ]:
#| export
for slug in pages:
    @rt(f'/{slug}')
    def page(htmx, slug:str=slug):
        title = slug.replace('-',' ').title()
        return layout(H2(title), from_md(pages[slug][1], img_dir='/public/pages'), 
                      title=f"{cfg.name} - {title}", htmx=htmx)